<a href="https://colab.research.google.com/github/schmitfe/Nawrot_CNS_Course/blob/main/Python_Programming_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introduction to Scientific Programming in Python - Day 4
# Iteration, Patterns, and Neural-like Dynamics


## Overview

In the nervous system, simple repeated processes generate complex behavior:
- **Neurons fire** in response to input, and their activity changes based on previous activity
- **Signals spread** through tissue via local interactions between neighbors
- **Activity patterns** emerge from the accumulation of many simple events
- **Memories** can be reconstructed from incomplete or noisy information through network interactions

In this module, we'll explore how **iteration** and **local interactions** create patterns. You will:
1. Iterate simple neural firing rules
2. Create fractals from random transformations
3. Model how activity spreads through tissue (cellular automata)
4. Implement a network that completes corrupted images (like memory recall)

We'll write the core algorithms, but the visualizations are provided so you can focus on understanding the dynamics.


---
## Setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
import scipy.signal

REPO_URL = "https://github.com/schmitfe/Nawrot_CNS_Course.git"
REPO_NAME = "Nawrot_CNS_Course"

def prepare_repo() -> Path:
    if "google.colab" in sys.modules:
        repo_root = Path("/content") / REPO_NAME
        if not repo_root.exists():
            subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
        os.chdir(repo_root)
        return repo_root.resolve()
    return Path.cwd().resolve()

REPO_ROOT = prepare_repo()
DATA_DIR = REPO_ROOT / "data" / "python_programming"
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR = {DATA_DIR}")


---
## Part 1: Neurons Responding to Input – Iteration in Action


### Biological Background
In the brain, a neuron's activity at time $t+1$ depends on:
- **Its previous activity** (how much does activity decay or persist?)
- **Synaptic input** it receives (signals from other neurons)

We can model this simply as:

$$\text{activity}(t+1) = \text{decay} \times \text{activity}(t) + \text{input}(t)$$

This is an example of **iteration**: each step depends on the previous one. When we loop this many times, we see how activity evolves.

**Your task:** Implement this update rule in a loop and observe what happens for different decay rates.


### Exercise 1.1: Firing Rate Dynamics


In [ ]:
# Create a time array of 100 time steps
n_steps = 100
activity = np.zeros(n_steps)
activity[0] = 0.1  # Initial firing rate

# External input: a pulse from time 20-40, then constant background input
external_input = np.ones(n_steps) * 0.1  # background
external_input[20:40] = 0.5  # pulse

# TODO: Fill in the decay parameter (what fraction of activity persists?)
# Try: 0.3 (fast decay), 0.7 (slow decay), 0.95 (very persistent)
decay = ???

# TODO: Fill in the input strength (how much input drives activity?)
# Try: 0.5 or 1.0
input_strength = ???

# TODO: Complete the loop to update activity at each time step
# Hint: activity[t+1] = decay * activity[t] + input_strength * external_input[t]
for t in range(n_steps - 1):
    activity[t+1] = ???  # Write the update rule here


### Visualization (provided)


In [ ]:
# Plot the activity trace
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

# Plot 1: Activity and input
ax1.plot(activity, 'b-', linewidth=2, label='Neural activity')
ax1.plot(external_input * 2, 'r--', linewidth=1.5, label='External input (scaled)')
ax1.set_ylabel('Activity (a.u.)')
ax1.set_xlabel('Time step')
ax1.set_title(f'Neuron Activity: decay={decay}, input_strength={input_strength}')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_ylim([0, max(activity) * 1.1])

# Plot 2: Phase portrait (activity vs. previous activity)
ax2.scatter(activity[:-1], activity[1:], s=20, alpha=0.6, color='darkblue')
ax2.plot([0, max(activity)], [0, max(activity)], 'k--', alpha=0.3, label='Identity')
ax2.set_xlabel('Activity(t)')
ax2.set_ylabel('Activity(t+1)')
ax2.set_title('Phase Portrait: Where Does Activity Go Next?')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()


### Questions to Answer

1. **Describe what you see:** How does the activity respond to the input pulse (time 20-40)? Does it rise quickly or slowly?

2. **Effect of decay:** What happens with decay=0.3 vs. decay=0.95? Which is "more forgetful"?

3. **Biological interpretation:** If this were a real neuron, what do you think decay represents biologically? (Hint: think about ion channels, membrane time constants)

Write 2-3 sentences here:


---
## Part 2: Random to Structure – Barnsley Fern


### Biological Background
How do plants form intricate patterns despite random cell division? How do brains develop structured connectivity despite noisy processes?

One answer: **randomness + iteration = structure**

The Barnsley fern is generated by:
1. Randomly pick one of several geometric transformations (weighted by probability)
2. Apply it to a point
3. Plot the point
4. Repeat thousands of times

What emerges? A beautiful fern, despite the randomness.

**Your task:** Implement the random selection and iterative application.


### Exercise 2.1: Generating the Barnsley Fern


In [ ]:
# Four affine transformations that define the Barnsley fern
def transform_1(point):
    """Stem (always selected with low probability)"""
    x, y = point
    return np.array([0.0, 0.16 * y])

def transform_2(point):
    """Serrated leaflet"""
    x, y = point
    return np.array([0.85 * x + 0.04 * y, -0.04 * x + 0.85 * y + 1.6])

def transform_3(point):
    """Smaller left leaflet"""
    x, y = point
    return np.array([0.2 * x - 0.26 * y, 0.23 * x + 0.22 * y + 1.6])

def transform_4(point):
    """Smallest left leaflet"""
    x, y = point
    return np.array([-0.15 * x + 0.28 * y, 0.26 * x + 0.24 * y + 0.44])

# List of all transformations
transformations = [transform_1, transform_2, transform_3, transform_4]

# TODO: Define the probabilities for each transformation
# These control which parts of the fern grow most
# They must sum to 1.0
# Standard values: [0.01, 0.85, 0.07, 0.07]
# Try: [0.05, 0.80, 0.07, 0.08] or other combinations
probabilities = np.array([???])  # Fill in 4 numbers that sum to 1.0

# Verify probabilities sum to 1
assert np.isclose(probabilities.sum(), 1.0), "Probabilities must sum to 1.0"

# TODO: How many iterations to run? More = finer detail
# Try: 100, 1000, 10000
n_iterations = ???

# Initialize array to store points
points = np.zeros((n_iterations, 2))
current_point = np.array([0.0, 0.0])

# TODO: Complete the loop
# At each iteration:
#   1. Randomly choose a transformation index using np.random.choice()
#   2. Apply that transformation to current_point
#   3. Store the resulting point
#   4. Update current_point
#
# Hint: np.random.choice(len(transformations), p=probabilities) picks an index
for i in range(n_iterations):
    # Choose which transformation to apply
    transform_idx = ???  # Use np.random.choice with probabilities

    # Apply the transformation
    current_point = transformations[transform_idx](current_point)

    # Store the point
    points[i] = current_point

print(f"Generated {n_iterations} points for the fern")


### Visualization (provided)


In [ ]:
# Plot the fern
fig, ax = plt.subplots(1, 1, figsize=(8, 10))

ax.scatter(points[:, 0], points[:, 1], s=0.5, c='green', alpha=0.5)
ax.set_aspect('equal')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Barnsley Fern ({n_iterations} iterations)\nProbabilities: {probabilities}')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


### Questions to Answer

1. **Pattern emergence:** Look at your fern. Describe its structure. Did randomness create deterministic structure?

2. **Effect of probabilities:** If you increase the probability of transform_2, what happens to the fern's shape?

3. **Biological connection:** How is this like how a neural network might form structure despite noisy/random synaptic growth?

Write 2-3 sentences here:


---
## Part 3: Local Interactions – Cellular Automata


### Biological Background
In neural tissue, each neuron influences its neighbors:
- Neurons that fire excite nearby neurons
- Activity can spread as waves through cortex
- Cardiac cells trigger each other in coordinated beats

A **cellular automaton** models this: each cell's next state depends only on its current state and its neighbors' states.

**Your task:** Implement the update rule that determines when a cell activates based on neighbor activity.


### Exercise 3.1: Spreading Activity in Tissue


In [ ]:
# Create a 2D grid representing neural tissue
grid_size = 50  # 50x50 grid
n_time_steps = 100

# Initialize grid: all cells resting (0) except center (active = 1)
grid = np.zeros((n_time_steps, grid_size, grid_size), dtype=int)
grid[0, grid_size//2, grid_size//2] = 1  # One neuron fires at center

# TODO: Choose activation threshold
# A cell becomes active if at least this many neighbors are active
# Try: 1 (spreads easily), 2 (moderate spread), 3 (hard to spread)
activation_threshold = ???


### Your Task: Implement the Update Function

Write a function that updates one time step of the cellular automaton.

For each cell in the grid:
- Count how many active neighbors it has (use the 4-neighbor von Neumann neighborhood: up, down, left, right)
- If neighbor_count >= activation_threshold, the cell becomes active
- Otherwise, the cell returns to resting state

Hint: Use `np.pad` to handle edges, or write a double loop for simplicity.


In [ ]:
def update_ca_step(current_grid, threshold):
    """
    Update one time step of cellular automaton.

    Args:
        current_grid: 2D array (grid_size x grid_size) with cell states (0 or 1)
        threshold: How many active neighbors needed to activate this cell?

    Returns:
        next_grid: Updated grid after one time step
    """

    next_grid = np.zeros_like(current_grid)
    grid_size = current_grid.shape[0]

    # TODO: For each cell, count neighbors and apply update rule
    #
    # Pseudocode:
    # for each cell (i, j) in the grid:
    #     Get neighbors (up, down, left, right)
    #     Count active neighbors
    #     if count >= threshold:
    #         next_grid[i,j] = 1
    #
    # You can get neighbors like: current_grid[i-1, j], current_grid[i+1, j], etc.
    # Handle edges: use % (modulo) for wraparound, or check bounds

    for i in range(grid_size):
        for j in range(grid_size):
            # Count active neighbors (4-neighbor von Neumann neighborhood)
            # Hint: neighbors are at (i±1, j) and (i, j±1)
            # Use modulo to wrap around edges: (i-1) % grid_size, etc.
            neighbors_active = ???  # Count from 4 neighbors

            # Apply update rule
            if neighbors_active >= threshold:
                next_grid[i, j] = 1
            else:
                next_grid[i, j] = 0

    return next_grid

# Run the simulation
for t in range(1, n_time_steps):
    grid[t] = update_ca_step(grid[t-1], activation_threshold)

print(f"Simulation complete with threshold={activation_threshold}")


### Visualization (provided)


In [ ]:
# Create heatmap showing grid evolution over time
fig, ax = plt.subplots(figsize=(10, 8))

# Transpose for better visualization (time on y-axis, space on x-axis)
im = ax.imshow(grid[:, :, grid_size//2], cmap='hot', aspect='auto', origin='lower')

ax.set_ylabel('Time step')
ax.set_xlabel('Space (center row)')
ax.set_title(f'Activity Spreading Through 2D Tissue (threshold={activation_threshold})')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Active (1) or Resting (0)')

plt.tight_layout()
plt.show()

# Also show a 2D snapshot of the final state
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for idx, t in enumerate([10, 50, n_time_steps-1]):
    axes[idx].imshow(grid[t], cmap='hot', origin='lower')
    axes[idx].set_title(f'Time step {t}')
    axes[idx].set_xlabel('X')
    axes[idx].set_ylabel('Y')

plt.tight_layout()
plt.show()


### Questions to Answer

1. **Pattern dynamics:** How does the activity spread when threshold=1 vs. threshold=3? Which spreads faster?

2. **Waves:** Do you see wave-like patterns? How would a real neural tissue with similar rules behave?

3. **Which threshold is most realistic?** For neural tissue, what would you expect? Why?

Write 2-3 sentences here:


---
## Part 4: Memory – Pattern Completion via Network Interactions


### Biological Background
Your brain can recognize a face from partial or corrupted visual input. Bad lighting, occlusion, noise—your brain still completes the pattern.

Theory suggests this happens via **attractor networks**: neurons that are strongly connected form patterns of activity. When you present a corrupted version, the network **settles** into the nearest stored pattern (like memory recall).

We'll model this with a simple update rule: each pixel updates toward the average of its neighbors, but also has an intrinsic "memory" of what it should be.

**Your task:** Implement the smoothing update rule that allows the network to recover an image.


### Exercise 4.1: Restoring a Corrupted Image


First, let's create a simple target image (a checkerboard pattern):


In [ ]:
# Create a simple target image: checkerboard
image_size = 32
target_image = np.zeros((image_size, image_size))
target_image[::2, ::2] = 1  # Checkerboard pattern
target_image[1::2, 1::2] = 1

# Corrupt it with noise
np.random.seed(42)
corruption_level = 0.3  # Flip 30% of pixels randomly
noisy_image = target_image.copy()
corrupt_indices = np.random.rand(image_size, image_size) < corruption_level
noisy_image[corrupt_indices] = 1 - noisy_image[corrupt_indices]

print(f"Target image: checkerboard pattern")
print(f"Corruption level: {corruption_level*100}%")


### Your Task: Implement the Update Rule

Write a function that updates the image using a smoothing rule combined with memory of the original noisy image:

For each pixel:
1. Look at its 4 neighbors (up, down, left, right)
2. Compute the average of neighbors
3. Update pixel as: new_value = smoothing_strength × neighbor_average + (1 - smoothing_strength) × original_noisy_value

This balances:
- **Smoothing:** Blending with neighbors helps remove noise
- **Memory:** Retaining the noisy input prevents over-smoothing


In [ ]:
def smooth_image_step(current_image, noisy_reference, smoothing_strength):
    """
    One step of image restoration via network smoothing.

    Args:
        current_image: Current state of the image
        noisy_reference: Original noisy image (reference to pull back toward)
        smoothing_strength: How much to blend with neighbors (0=no smoothing, 1=full smoothing)

    Returns:
        updated_image: Image after one smoothing step
    """

    updated_image = np.zeros_like(current_image, dtype=float)
    h, w = current_image.shape

    # TODO: For each pixel, apply the smoothing rule
    #
    # Pseudocode:
    # for each pixel (i, j):
    #     1. Get values of 4 neighbors (with wraparound via modulo)
    #     2. Compute neighbor_average
    #     3. updated_image[i,j] = smoothing_strength * neighbor_average +
    #                              (1 - smoothing_strength) * noisy_reference[i,j]
    #
    # This is very similar to the CA exercise!

    for i in range(h):
        for j in range(w):
            # Get 4 neighbors (wraparound at edges)
            neighbor_values = [
                current_image[(i-1) % h, j],  # up
                current_image[(i+1) % h, j],  # down
                current_image[i, (j-1) % w],  # left
                current_image[i, (j+1) % w],  # right
            ]
            neighbor_average = ???  # Compute mean of neighbors

            # Apply smoothing rule: blend neighbors with memory of noisy original
            updated_image[i, j] = smoothing_strength * neighbor_average + \
                                  (1 - smoothing_strength) * noisy_reference[i, j]

    return updated_image

# TODO: Choose smoothing strength
# 0.1 = mostly remember noisy input (little smoothing)
# 0.5 = equal blend
# 0.9 = mostly smooth with neighbors (more smoothing)
smoothing_strength = ???

# TODO: How many iterations?
# Try 10, 50, 100
n_iterations = ???

# Run restoration
restored_images = [noisy_image.copy()]
current_image = noisy_image.copy()

for iteration in range(n_iterations):
    current_image = smooth_image_step(current_image, noisy_image, smoothing_strength)
    if iteration in [5, 25, n_iterations-1] or iteration < 5:  # Store some intermediate steps
        restored_images.append(current_image.copy())

print(f"Restoration complete: {len(restored_images)} snapshots saved")


### Visualization (provided)


In [ ]:
# Show progression of restoration
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

steps_to_show = [0, 5, 25, n_iterations-1]
for idx, step in enumerate(steps_to_show):
    if step < len(restored_images):
        im = axes[idx].imshow(restored_images[step], cmap='gray', vmin=0, vmax=1)
        axes[idx].set_title(f'Iteration {step}')
        axes[idx].axis('off')

plt.suptitle(f'Image Restoration: smoothing_strength={smoothing_strength}', y=1.02)
plt.tight_layout()
plt.show()

# Compare original and final restored
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].imshow(target_image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Target (clean)')
axes[0].axis('off')

axes[1].imshow(noisy_image, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Corrupted ({corruption_level*100}% noise)')
axes[1].axis('off')

axes[2].imshow(current_image, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Restored\n(strength={smoothing_strength})')
axes[2].axis('off')

# Compute reconstruction error
error = np.mean(np.abs(current_image - target_image))
axes[3].text(0.1, 0.5, f'Reconstruction Error:\n{error:.4f}', fontsize=14, transform=axes[3].transAxes)
axes[3].axis('off')

plt.tight_layout()
plt.show()


### Questions to Answer

1. **Smoothing strength effects:**
   - What happens with smoothing_strength=0.1? (mostly preserve noisy input)
   - What happens with smoothing_strength=0.9? (mostly smooth with neighbors)
   - Which recovers the original pattern best?

2. **Memory metaphor:** How is this like a brain recalling a memory from partial/corrupted information? What role do the "neighbors" (connected neurons) play?

3. **Biological plausibility:** In a real neural network, what would "neighbors" represent? (connections between neurons) How could this happen biologically?

Write 3-4 sentences here:


---
## Summary and Reflection

In this module, you implemented three core concepts in computational neuroscience:

1. **Iteration** – Neural activity evolving based on previous activity and input
2. **Emergence** – Complex patterns arising from random processes and simple rules
3. **Collective dynamics** – Local interactions (between neighbors) generating global patterns
4. **Memory and computation** – Networks settling into patterns to recall information

These concepts appear everywhere in neuroscience:
- In the motor cortex neurons you'll analyze in later modules
- In how the brain perceives things (completing partial information)
- In pattern formation during development
- In how diseases like epilepsy spread through tissue (abnormal CA dynamics)


### Final Questions

1. Which part felt most like "real neuroscience" to you? Why?

2. If you wanted to make these models more realistic, what would you change?

3. How could you use these ideas to understand data from actual neural recordings (like in the Neural Data Analysis module)?

Write 3-4 sentences for each:
